# pytempo: a guided tour

`pytempo` reads Romanian official statistics from the INS TEMPO Online API:
it finds indicators, reads their metadata, pulls the data into a pandas
DataFrame, reshapes it, and writes the SQL to load it into PostgreSQL.

**This notebook runs live against the INS server.** Every cell makes real
requests, so it needs a network connection, and a few cells take a few
seconds. The examples were chosen to stay small and polite: no cell here
makes more than a couple of requests.

Install it straight from GitHub:

    pip install git+https://github.com/CIDS-UBB/pytempo.git


In [1]:
import pytempo as t

## 1. Discovery

Start with a panorama: how big the catalogue is and where to begin.


In [2]:
t.overview()

pytempo: 1916 TEMPO indicators, in 8 top level domains.
Start with find('salariati') or domains(). t.help() has the full guide.


`find` is the plain keyword search. It looks in the indicator name and in
its code, ignores diacritics and case, and requires every word you give it to
match. It returns **all** matches, not a truncated page, so slice the result
if you only want a few.


In [3]:
results = t.find("salariati")
print(len(results), "indicators")
results[:5]

104 indicators


[Matrix('AMG1103', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe grupe de varsta si sexe'),
 Matrix('AMG1104', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe grupe de varsta si medii de rezidenta'),
 Matrix('AMG1105', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe activitati si sexe'),
 Matrix('AMG1106', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe activitati si medii de rezidenta'),
 Matrix('AMG115K', 'AMIGO - Salariati cu regim de lucru temporar dupa durata obisnuita a saptamanii de lucru si sexe')]

`search` is the other tool: discovery with filters. The keyword is optional,
and the filters combine with each other.

* `domeniu` matches a substring of the statistical domain name, so `economic`
  finds `B. STATISTICA ECONOMICA` without you knowing the exact wording.
* `periodicitate` matches a substring of how often the indicator is published.
* `level` keeps only indicators that reach that territorial level.
* `caen=True` keeps only those with a CAEN activity classification.

The difference in one line: **`find` searches names, `search` filters on
metadata.**


In [4]:
economic = t.search(domeniu="economic", periodicitate="anuala", level="judet")
print(len(economic), "indicators")
economic[:5]

111 indicators


[Matrix('AGR101A', 'Suprafata fondului funciar dupa modul de folosinta, pe forme de proprietate, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR101B', 'Suprafata fondului funciar dupa modul de folosinta, pe judete si localitati'),
 Matrix('AGR102A', 'Suprafata terenurilor amenajate cu lucrari de irigatii si suprafata agricola irigata, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR102B', 'Suprafata terenurilor amenajate cu lucrari de desecare, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR102C', 'Suprafata terenurilor amenajate cu lucrari de ameliorare si combaterea eroziunii solului, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete')]

`t.filters()` prints what you can filter on, with the real values read from
the catalogue rather than from a hardcoded list.


In [5]:
t.filters()

Filters for t.search(). They combine with each other and with the search words.
  level        : ['national', 'macroregiune', 'regiune', 'judet', 'localitate', 'necunoscut']
  caen         : True only those with a CAEN dimension, False only those without
  domeniu      : substring of the domain name, diacritics ignored
                 A. STATISTICA SOCIALA
                 B. STATISTICA ECONOMICA
                 C. FINANTE
                 D. JUSTITIE
                 E. MEDIU INCONJURATOR
                 F. UTILITATI PUBLICE SI ADMINISTRAREA TERITORIULUI
                 G. DEZVOLTARE DURABILA - Orizont 2020
                 H. DEZVOLTARE DURABILA - Tinte 2030
  periodicitate: substring of the periodicity, diacritics ignored
                 ['5 - 6 ani', 'Anuala', 'Cincinal', 'La 2 ani', 'La fiecare 3 ani sau mai mult', 'La trei ani', 'Lunara', 'Perioada neregulata', 'Recomandabil la 10 ani', 'Trimestriala']

The metadata filters rest on the local index. If it is missing,
search a

## 2. Understanding one indicator

Before pulling data, read what the indicator actually is. We use FOM104D,
the average number of employees by county and locality.


In [6]:
m = t.matrix("FOM104D")
m

Matrix('FOM104D', 'Numarul mediu al salariatilor pe judete si localitati')

`what()` is the short version: the first sentence of the definition, the
unit, how often it is published, when it was last updated, and a warning if
the observations mention specific years, which usually signals a break in
the series.


In [7]:
m.what()

FOM104D  Numarul mediu al salariatilor pe judete si localitati
  Numarul mediu al salariatilor cuprinde persoanele angajate cu contract de munca/raport de serviciu pe durata determinata sau nedeterminata (inclusiv lucratorii sezonieri, managerul sau administratorul), al caror contract de munca/raport de serviciu nu a fost suspendat in perioada de referinta.
  unit        : Numar persoane
  periodicity : Anuala
  updated     : 20-11-2025
                read them with .describe()


`where()` shows where the indicator sits in the domain tree, and what it
covers: how many territorial units at each level, whether localities carry a
SIRUTA code, and the span of years.

A note on **territorial levels**. `national`, `macroregiune`, `regiune`,
`judet`, `localitate` are how pytempo interprets the option names, not a
concept INS exposes directly. A territorial dimension usually mixes all of
them in one column, and pytempo works out which is which so you can ask for
one. Names that do not fit the administrative nomenclator, such as monitoring
stations, are labelled `necunoscut` rather than being forced into a level.


In [8]:
m.where()

domain   : A. STATISTICA SOCIALA > FORTA DE MUNCA > SALARIATI
territory: Judete (43 options)
    national        1
    judet           42
territory: Localitati (3183 options)
    localitate      3183
SIRUTA   : yes
time     : Ani, 35 periods, 1990 to 2024


`how()` generates the download manual for this specific indicator: the
commands that make sense for it, the strategy that will be used, and how many
requests to expect.

It also says which of the two ways of pulling the data this indicator needs,
`get()` or `download()`, and prints the command ready to copy.


In [9]:
m.how()

How to download FOM104D:


  m = t.matrix('FOM104D')
  df = m.get()          level localitate, tidied
  m.get(level='national')
  m.get(level='judet')
  m.get(level=None)     every level at once

  county and locality are separate dimensions here, so a level picks
  which one is active and puts the other on TOTAL: level='judet' gives
  one row per county in a single request, level='localitate' gives the
  localities, county by county.
  m.get(raw=True)       exactly what INS returns, no extras

  strategy: by_county, roughly 43 requests
  downloaded in several requests and concatenated


`describe()` prints the full record exactly as INS wrote it: the complete
definition, the methodology, the sources and the observations. It is long on
purpose. The observations are where series breaks and warnings about
incomplete years live, so read them before trusting a series.


In [10]:
m.describe()

FOM104D  Numarul mediu al salariatilor pe judete si localitati
domain      : A. STATISTICA SOCIALA > FORTA DE MUNCA > SALARIATI
levels      : national, judet, localitate
periodicity : Anuala
updated     : 20-11-2025

DEFINITION
Numarul mediu al salariatilor cuprinde persoanele angajate cu contract de munca/raport de serviciu pe durata determinata sau nedeterminata (inclusiv lucratorii sezonieri, managerul sau administratorul), al caror contract de munca/raport de serviciu nu a fost suspendat in perioada de referinta.
Numarul mediu al salariatilor se calculeaza ca medie aritmetica simpla rezultata din suma efectivelor zilnice de salariati (exclusiv cei al caror contract de munca/raport de serviciu a fost suspendat), din perioada de referinta, inclusiv din zilele de repaus saptamanal, sarbatori legale si alte zile nelucratoare, impartita la numarul total al zilelor calendaristice.
In efectivul zilnic al salariatilor luat in calculul numarului mediu se cuprind urmatoarele categorii:
- sal

`options()` with no argument lists the dimensions, each with the role
pytempo assigned to it and how many values it has. With an argument it lists
the values of one dimension, named by label, role, index or level.


In [11]:
m.options()

[0] Judete (teritoriu, 43 options)
[1] Localitati (teritoriu, 3183 options)
[2] Ani (timp, 35 options)
[3] UM: Numar persoane (um, 1 options)

## 3. A simple extraction

FOM101A, labour resources by county, fits in a single request, so it is a
good first pull.

`get()` with no arguments does three things by default: it picks the finest
territorial level the indicator actually reaches, it applies the tidy
standardization, and it prints one line saying what it decided. When that
default leaves coarser levels out, it names them and tells you how to get
them back.


In [12]:
df = t.matrix("FOM101A").get()
df.shape

FOM101A: level judet (the finest), single, 1 request
  for every level, including national, macroregiune and regiune, use get(level=None)


(4392, 7)

The result is in **long format**: one row per combination, one text column
per dimension, a numeric `Valoare` column, and then the derived columns that
tidy added.

Tidy only adds the columns that carry something. A county dimension has no
SIRUTA code and no settlement type, so it gets only its level column.


In [13]:
df.head()

,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",Ani,UM: Mii persoane,Valoare,"Macroregiuni, regiuni de dezvoltare si judete_nivel",Ani_an
0,Total,Arges,Anul 1990,Mii persoane,394.8,judet,1990
1,Total,Arges,Anul 1991,Mii persoane,394.7,judet,1991
2,Total,Arges,Anul 1992,Mii persoane,399.5,judet,1992
3,Total,Arges,Anul 1993,Mii persoane,392.9,judet,1993
4,Total,Arges,Anul 1994,Mii persoane,392.2,judet,1994


The derived columns are typed, not text: `Int64` for the year, nullable
strings for the level.


In [14]:
df.dtypes

Sexe                                                       str
Macroregiuni, regiuni de dezvoltare si judete              str
Ani                                                        str
UM: Mii persoane                                           str
Valoare                                                float64
Macroregiuni, regiuni de dezvoltare si judete_nivel     string
Ani_an                                                   Int64
dtype: object

Asking for a level changes what comes back. Counties and regions are
different slices of the same indicator, so the row counts differ.


In [15]:
counties = t.matrix("FOM101A").get(level="judet", progress=False)
regions = t.matrix("FOM101A").get(level="regiune", progress=False)
print("judet  :", counties.shape)
print("regiune:", regions.shape)

judet  : (4392, 7)
regiune: (840, 7)


`raw=True` gives exactly what INS returned, with no derived columns. Use raw
when you want to see the source untouched or you are writing your own
processing; use tidy, the default, when you want to work with the data.


In [16]:
raw = t.matrix("FOM101A").get(raw=True, progress=False)
print("raw :", raw.shape)
print("tidy:", df.shape)
raw.head(3)

raw : (4392, 5)
tidy: (4392, 7)


,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",Ani,UM: Mii persoane,Valoare
0,Total,Arges,Anul 1990,Mii persoane,394.8
1,Total,Arges,Anul 1991,Mii persoane,394.7
2,Total,Arges,Anul 1992,Mii persoane,399.5


## 4. Standardization and SIRUTA

SIRUTA is the official code of a Romanian administrative unit. INS puts it
inside the locality name, as a numeric prefix, so the raw label looks like
`1017 MUNICIPIUL ALBA IULIA`.

pytempo splits that into separate columns and **keeps the original label
untouched**. SIRUTA is preserved as a key, never dropped, because it is what
lets you join this data with other administrative sources: population
registers, budgets, geographies.

We use SAN103B here, children enrolled in nurseries by county and locality,
because it reaches locality level and still fits in a single request.


In [17]:
nurseries = t.matrix("SAN103B").get()
localities = nurseries[nurseries["Localitati_nivel"] == "localitate"]
localities[["Localitati", "Localitati_siruta", "Localitati_tip",
            "Localitati_nume", "Valoare"]].head(6)

SAN103B: all levels, single, 1 request


,Localitati,Localitati_siruta,Localitati_tip,Localitati_nume,Valoare
2,1017 MUNICIPIUL ALBA IULIA,1017,municipiu,ALBA IULIA,50
3,1213 MUNICIPIUL AIUD,1213,municipiu,AIUD,41
5,9262 MUNICIPIUL ARAD,9262,municipiu,ARAD,159
6,9459 ORAS CHISINEU-CRIS,9459,oras,CHISINEU-CRIS,14
7,9538 ORAS INEU,9538,oras,INEU,18
8,9574 ORAS LIPOVA,9574,oras,LIPOVA,50


Notice what happened: the code, the type of settlement and the clean name are
now three separate, typed columns, while the `Localitati` column still holds
the original text.

One more thing to know: **the data is sparse.** Combinations with no data are
absent as whole rows, they do not arrive as `NaN`. That reflects real
administrative history rather than a gap in the library: Ilfov and Municipiul
Bucuresti do not exist as separate units before 1996, so those rows simply do
not exist. Do not check a download by comparing the row count against the
product of the dimensions.


In [18]:
print("rows returned :", len(nurseries))
print("empty values  :", int(nurseries["Valoare"].isna().sum()))

rows returned : 180
empty values  : 0


## 5. Two territorial dimensions

Some indicators keep county and locality as **two separate dimensions**.
FOM104D is one: it has `Judete` with 43 options and `Localitati` with 3183.

There, a level picks which dimension is active and puts the other on its
total. `level='judet'` gives one row per county, with localities pinned to
TOTAL, in a single request. `level='localitate'` gives the localities, and
because the data is keyed by the real county and locality pair, the county
dimension stays whole and the download goes county by county.

We run the cheap one here.


In [19]:
by_county = t.matrix("FOM104D").get(level="judet")
print(by_county.shape)
by_county.head(3)

FOM104D: level judet, single, 1 request
  for every level, including national and localitate, use get(level=None)


(1469, 8)


,Judete,Localitati,Ani,UM: Numar persoane,Valoare,Judete_nivel,Localitati_nivel,Ani_an
0,Alba,TOTAL,Anul 1990,Numar persoane,149181,judet,national,1990
1,Alba,TOTAL,Anul 1991,Numar persoane,137228,judet,national,1991
2,Alba,TOTAL,Anul 1992,Numar persoane,129564,judet,national,1992


## 6. A larger download, with automatic splitting

**This cell takes longer than the others.** It makes more than one request.

A single POST to INS is capped at a cell budget. When an indicator is larger
than that, pytempo splits the work automatically and concatenates the pieces:
county by county for indicators that reach locality level, otherwise on the
largest dimension. You do not have to plan anything, but you should know it is
happening, which is why `get()` prints the decision line.

FOM106E splits on the CAEN dimension into a couple of requests. Indicators
that would take hundreds of requests are not used in this tutorial: past 50
requests `get()` stops and sends you to `download()`, which is the subject of
the next cell.


In [20]:
big = t.matrix("FOM106E").get()
big.shape

FOM106E: level judet (the finest), split:CAEN Rev.2  (activitati ale economiei nationale), 2 requests
  for every level, including national, macroregiune and regiune, use get(level=None)


  1/2: +84354 rows (total 84354)


  2/2: +45256 rows (total 129610)


(129610, 8)

### When splitting is not enough: download()

FOM106E was a couple of requests. Some indicators are hundreds: POP107D at
locality level is 380 of them, and `get()` refuses to start it. Not out of
caution. `get()` keeps every request in memory until the last one comes back,
so one late timeout, and INS does time out, loses the whole download with
nothing to resume from. Measured on SAN101B: five hours through `get()` and
abandoned, under three minutes writing each county to disk.

`download()` is the same call for that case. Same `level`, `levels` and
`select`, same plan, built by the same code. What changes is where the answers
go: each request is written to its own slice file the moment it arrives.

```python
m = t.matrix("POP107D")
m.how()          # says this one is large, and prints the command to copy

df = m.download(level="localitate", folder="data/pop107d")
```

Memory stays at one request, an interrupted run keeps what it had, and running
the same call again asks only for the slices that are not on disk yet. A
request that keeps failing is reported at the end instead of sinking the rest,
and the join is checked before you get the frame: nothing missing, no rows lost
or doubled, no combination of dimensions occurring twice. `return_df=False`
returns the path of the CSV instead of the frame, for indicators too large to
hold at all.

Often the cheaper answer is not to download all of it. `how()` also names the
dimensions worth trimming: POP107D has 104 ages, and asking for the two you
need with `select=` turns 380 requests into a handful.

Nothing is run here: a 380 request download does not belong in a tutorial.


## 7. Reshaping: the df.tempo accessor

Any frame from `get(tidy=True)` carries a `df.tempo` accessor. It reshapes
and summarizes, never fetches anything, and never changes the frame you give
it. No extra requests are made below: we reuse the FOM101A frame.

`coverage()` is the first look at a series: one row per territorial unit, the
span of years it has, how many of the years seen anywhere in the frame are
missing for it, and the smallest and largest value with the year each
occurred.


In [21]:
df.tempo.coverage().head(8)

,"Macroregiuni, regiuni de dezvoltare si judete",first_year,last_year,n_years,missing_years,min_value,min_year,max_value,max_year
0,Alba,1990,2024,35,0,92.1,2019,246.1,1990
1,Arad,1990,2024,35,0,123.1,2022,299.3,2011
2,Arges,1990,2024,35,0,168.4,2022,412.5,2011
3,Bacau,1990,2024,35,0,165.5,2019,473.7,2010
4,Bihor,1990,2024,35,0,168.0,2022,391.4,1990
5,Bistrita-Nasaud,1990,2024,35,0,80.2,2019,209.7,2010
6,Botosani,1990,2024,35,0,105.4,2019,280.0,2010
7,Braila,1990,2024,35,0,78.4,2022,238.9,2005


`wide()` pivots time into columns, which is the shape you would put in a
paper. The index is built from the original dimension columns, leaving out
the derived ones, the original time column, which says the same thing as the
year, and a unit of measure column that never varies.


In [22]:
wide = df.tempo.wide()
print(wide.shape)
wide.iloc[:4, :8]

(129, 37)


,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",1990,1991,1992,1993,1994,1995
0,Feminin,Alba,116.5,113.0,114.5,116.8,117.2,113.1
1,Feminin,Arad,140.8,135.2,133.5,131.7,129.5,134.8
2,Feminin,Arges,191.4,191.2,200.3,196.5,195.8,194.0
3,Feminin,Bacau,197.9,194.2,198.1,198.6,202.1,212.0


`df.tempo.geo()` is a documented stub. It will join on SIRUTA and return a
GeoDataFrame, arriving as an optional `pytempo[geo]` extra so that anyone who
only wants the numbers never pays for the geometry stack.

Calling the accessor on a frame that is not tidy output says so plainly
rather than guessing.


In [23]:
try:
    df.tempo.geo()
except NotImplementedError as e:
    print(e)

geo() is not implemented yet: it will join on SIRUTA and return a GeoDataFrame, arriving as the optional pytempo[geo] extra


## 8. A small analysis

The data comes back ready to use. Nothing below is pytempo specific: from
here on it is ordinary pandas.


In [24]:
terr = "Macroregiuni, regiuni de dezvoltare si judete"
cluj = df[(df[terr] == "Cluj") & (df["Sexe"] == "Total")]
cluj = cluj.sort_values("Ani_an")
cluj[[terr, "Ani_an", "Valoare"]].tail(10)

,"Macroregiuni, regiuni de dezvoltare si judete",Ani_an,Valoare
305,Cluj,2015,464.4
306,Cluj,2016,471.0
307,Cluj,2017,468.8
308,Cluj,2018,464.9
309,Cluj,2019,465.5
310,Cluj,2020,467.8
311,Cluj,2021,472.0
312,Cluj,2022,437.1
313,Cluj,2023,443.7
314,Cluj,2024,447.2


No plots here, on purpose: this notebook adds no dependencies beyond what
pytempo already needs. Plotting, mapping and modelling are ordinary work on
an ordinary DataFrame.


In [25]:
cluj["Valoare"].describe()

count     35.000000
mean     456.131429
std       10.760597
min      434.300000
25%      447.450000
50%      459.500000
75%      464.650000
max      472.000000
Name: Valoare, dtype: float64

## 9. Loading into PostgreSQL

pytempo never connects to a database. It writes the SQL as text, and you
decide how to run it. That keeps the dependencies at requests and pandas.

`m.schema()` generates the CREATE TABLE for one indicator: one text column per
dimension, a numeric value, and exactly the derived columns that
`get(tidy=True)` produces for it. Nothing is guessed twice, so the table
cannot drift away from the DataFrame.


In [26]:
print(t.matrix("FOM101A").schema())

CREATE TABLE IF NOT EXISTS tempo.fom101a (
    sexe text,
    macroregiuni_regiuni_de_dezvoltare_si_judete text,
    ani text,
    um_mii_persoane text,
    valoare numeric,
    macroregiuni_regiuni_de_dezvoltare_si_judete_nivel text,
    ani_an smallint
);

COMMENT ON TABLE tempo.fom101a IS 'Resurse de munca pe sexe, macroregiuni, regiuni de dezvoltare si judete. Resursele de munca la 1 ianuarie reprezinta acea categorie de populatie care dispune de ansamblul capacitatilor fizice si intelectuale care ii permit sa desfasoare o munca utila in una din activitatile economie nationale';

COMMENT ON COLUMN tempo.fom101a.valoare IS 'Measured in Mii persoane';

CREATE INDEX IF NOT EXISTS fom101a_ani_an_idx ON tempo.fom101a (ani_an);



`t.schema_catalog()` generates the shared infrastructure: `indicators` and
`dimensions` describe the catalogue, and `territory` is a SIRUTA lookup you
fill from the data you extract.


In [27]:
print(t.schema_catalog())

CREATE SCHEMA IF NOT EXISTS tempo;

CREATE TABLE IF NOT EXISTS tempo.indicators (
    code text PRIMARY KEY,
    name text NOT NULL,
    domain text,
    family text,
    periodicity text,
    last_updated text,
    total_cells bigint,
    has_siruta boolean
);

COMMENT ON TABLE tempo.indicators IS 'One row per TEMPO indicator, from the pytempo registry.';

CREATE TABLE IF NOT EXISTS tempo.dimensions (
    code text NOT NULL REFERENCES tempo.indicators (code),
    position smallint NOT NULL,
    label text NOT NULL,
    role text,
    n_options integer,
    PRIMARY KEY (code, position)
);

COMMENT ON TABLE tempo.dimensions IS 'The dimensions of each indicator, in dimensionsMap order.';

CREATE TABLE IF NOT EXISTS tempo.territory (
    siruta integer PRIMARY KEY,
    name text NOT NULL,
    kind text,
    county text
);

COMMENT ON TABLE tempo.territory IS 'SIRUTA lookup, filled from the data you extract.';

CREATE INDEX IF NOT EXISTS territory_county_idx ON tempo.territory (county);



`t.column_mapping(m)` gives the mapping from DataFrame column names to SQL
identifiers, so renaming before loading is one line. The full pipeline is:

    open("catalog.sql", "w").write(t.schema_catalog())
    open("fom101a.sql", "w").write(m.schema())
    # psql -f catalog.sql -f fom101a.sql

    df = df.rename(columns=t.column_mapping(m))
    df.to_sql("fom101a", engine, schema="tempo", if_exists="append",
              index=False)


In [28]:
t.column_mapping(t.matrix("FOM101A"))

{'Sexe': 'sexe',
 'Macroregiuni, regiuni de dezvoltare si judete': 'macroregiuni_regiuni_de_dezvoltare_si_judete',
 'Ani': 'ani',
 'UM: Mii persoane': 'um_mii_persoane',
 'Valoare': 'valoare',
 'Macroregiuni, regiuni de dezvoltare si judete_nivel': 'macroregiuni_regiuni_de_dezvoltare_si_judete_nivel',
 'Ani_an': 'ani_an'}

## Where to go next

* `t.help()` prints the full navigation guide.
* `m.help()` does the same for one indicator.
* `m.how()` gives you the download commands for that specific indicator.
* The README covers levels, roles, the shape of the data, the wrangling
  accessor, the PostgreSQL bridge and the internal schema registry.

One closing thought. pytempo does one job: getting the data out of TEMPO,
correctly and reproducibly, and handing it over in a shape you can work with.
Analysis, visualisation and mapping are not its job, and it deliberately stays
out of the way so you can use the usual tools.
